In [2]:
import sys
from pathlib import Path

PROJECT_ROOT =  Path('/content/drive/MyDrive/MVA/NLP')

sys.path.append(str(PROJECT_ROOT))
print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Exists: {PROJECT_ROOT.exists()}")

PROJECT_ROOT = /content/drive/MyDrive/MVA/NLP
Exists: True


In [3]:
!pip install -q jiwer soundfile torchaudio scikit-learn seaborn matplotlib

In [4]:
import json
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from jiwer import cer

from src.data import (
    parse_transcript_file,
    fix_audio_extension,
    ASRDataset,
    build_vocab_from_df,
    make_collate_fn,
)
from src.model_baseline import HuBERT_CTC

from src.train_utils import (
    set_seed,
    train_epoch,
    evaluate,
    save_checkpoint,
    count_parameters,
)
from src.eval_utils import (
    run_inference,
    save_results_json,
    save_predictions_csv,
)

In [5]:
EXPERIMENT = "multilingual"   # "monolingual" or "multilingual"


CONFIG = {
    "model_name": "utter-project/mHuBERT-147-base-3rd-iter",
    "data_size": "10min",
    "lang": "swa",
    "batch_size": 8,
    "grad_accumulation": 4,
    "learning_rate": 1e-4,
    "weight_decay": 1e-6,
    "target_iterations": 1500,
    "eval_every": 5,
    "save_every": 100,
    "experiment_name": f"mhubert_{EXPERIMENT}_swa_10min",
}

TRAIN_LANGUAGES = ["swa"] if EXPERIMENT == "monolingual" else ["swa", "amh", "wol"]

In [6]:
SWA_DIR  = PROJECT_ROOT / "data" / "commonvoice" / "swa"
DATA_DIR = PROJECT_ROOT / "data" / "commonvoice"

RESULTS_DIR     = PROJECT_ROOT / "results"     / "mono_vs_multi"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints" / "mono_vs_multi"
FIGURES_DIR     = PROJECT_ROOT / "figures"     / "mono_vs_multi"

for d in [RESULTS_DIR, CHECKPOINTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [7]:
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [8]:
import pandas as pd
from torch.utils.data import ConcatDataset


df_dev  = fix_audio_extension(parse_transcript_file(SWA_DIR / "transcript_10min_dev.txt"))
df_test = fix_audio_extension(parse_transcript_file(SWA_DIR / "transcript_10min_test.txt"))


train_dfs = []
for lang in TRAIN_LANGUAGES:
    lang_dir = DATA_DIR / lang
    df = fix_audio_extension(
        parse_transcript_file(lang_dir / f"transcript_10min_train.txt")
    )
    df["audio_dir"] = str(lang_dir / "wav")
    train_dfs.append(df)

df_train = pd.concat(train_dfs, ignore_index=True)
print(f"Train : {len(df_train)} utterances ({', '.join(TRAIN_LANGUAGES)})")
print(f"Dev   : {len(df_dev)} utterances (swa)")
print(f"Test  : {len(df_test)} utterances (swa)")


vocab       = build_vocab_from_df(df_train)
idx_to_char = {v: k for k, v in vocab.items()}
print(f"Vocab : {len(vocab)} tokens")

Parsed 171 samples from /content/drive/MyDrive/MVA/NLP/data/commonvoice/swa/transcript_10min_dev.txt
Parsed 175 samples from /content/drive/MyDrive/MVA/NLP/data/commonvoice/swa/transcript_10min_test.txt
Parsed 175 samples from /content/drive/MyDrive/MVA/NLP/data/commonvoice/swa/transcript_10min_train.txt
Parsed 97 samples from /content/drive/MyDrive/MVA/NLP/data/commonvoice/amh/transcript_10min_train.txt
Parsed 142 samples from /content/drive/MyDrive/MVA/NLP/data/commonvoice/wol/transcript_10min_train.txt
Train : 414 utterances (swa, amh, wol)
Dev   : 171 utterances (swa)
Test  : 175 utterances (swa)
Vocab : 212 tokens


In [9]:
AUDIO_DIR_SWA = SWA_DIR / "wav"
dev_dataset   = ASRDataset(df_dev,  AUDIO_DIR_SWA)
test_dataset  = ASRDataset(df_test, AUDIO_DIR_SWA)


from torch.utils.data import ConcatDataset

if EXPERIMENT == "monolingual":
    train_dataset = ASRDataset(df_train, SWA_DIR / "wav")
else:
    train_datasets = []
    for lang in TRAIN_LANGUAGES:
        df_lang = df_train[df_train["audio_dir"].str.contains(lang)].copy()
        train_datasets.append(ASRDataset(df_lang, DATA_DIR / lang / "wav"))
    train_dataset = ConcatDataset(train_datasets)

print(f"Train dataset : {len(train_dataset)} samples")

Loaded 171 samples from                utt_id        audio_filename  \
0    ALFFA_swa_000175  ALFFA_swa_000175.wav   
1    ALFFA_swa_000176  ALFFA_swa_000176.wav   
2    ALFFA_swa_000177  ALFFA_swa_000177.wav   
3    ALFFA_swa_000178  ALFFA_swa_000178.wav   
4    ALFFA_swa_000179  ALFFA_swa_000179.wav   
..                ...                   ...   
166  ALFFA_swa_000341  ALFFA_swa_000341.wav   
167  ALFFA_swa_000342  ALFFA_swa_000342.wav   
168  ALFFA_swa_000343  ALFFA_swa_000343.wav   
169  ALFFA_swa_000344  ALFFA_swa_000344.wav   
170  ALFFA_swa_000345  ALFFA_swa_000345.wav   

                                                  text  
0    tutakapo kuwa na mjadala mwingine kama huu lab...  
1    ziko zinaingia kwenye machafuko nurdin ni hayo...  
2    anadai kuwa madai ya ocampo kuwa amefanya uchu...  
3       alifunguliwa mashtaka mwaka elfu mbili na sita  
4    hizi ni materials ambazo zinatengenezwa kwenye...  
..                                                 ...  
166  eeh mi 

In [10]:
collate_fn = make_collate_fn(vocab)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
)

In [14]:
num_train_samples = len(train_dataset)
effective_batch = CONFIG["batch_size"] * CONFIG["grad_accumulation"]
iterations_per_epoch = num_train_samples / effective_batch
num_epochs = int((CONFIG["target_iterations"] / iterations_per_epoch) + 0.9999)

CONFIG["num_epochs"] = num_epochs
print("num_epochs =", num_epochs)

num_epochs = 116


In [11]:
model = HuBERT_CTC(
    vocab_size=len(vocab),
    model_name=CONFIG["model_name"],
).to(device)

count_parameters(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Total params: 103.6M
Trainable params: 9.22M (8.9%)
Frozen params: 94.4M


(103595361, 9223649)

In [12]:
criterion = nn.CTCLoss(blank=vocab["<blank>"], zero_infinity=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=10,
)

In [15]:
best_dev_loss = float("inf")
history = {
    "train_loss": [],
    "dev_loss": [],
    "lr": [],
    "epoch": [],
}

start_time = time.time()
total_iterations = 0
best_ckpt_path = CHECKPOINTS_DIR / f"{CONFIG['experiment_name']}_best.pt"

for epoch in range(CONFIG["num_epochs"]):
    train_loss = train_epoch(
        model, train_loader, criterion, optimizer,
        device, CONFIG["grad_accumulation"]
    )

    total_iterations += len(train_loader) // CONFIG["grad_accumulation"]

    if (epoch + 1) % CONFIG["eval_every"] == 0 or epoch == 0 or epoch == CONFIG["num_epochs"] - 1:
        dev_loss = evaluate(model, dev_loader, criterion, device)
        scheduler.step(dev_loss)
    else:
        dev_loss = history["dev_loss"][-1] if history["dev_loss"] else float("inf")

    history["train_loss"].append(train_loss)
    history["dev_loss"].append(dev_loss)
    history["lr"].append(optimizer.param_groups[0]["lr"])
    history["epoch"].append(epoch + 1)

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            path=best_ckpt_path,
            dev_loss=dev_loss,
            config=CONFIG,
            vocab=vocab,
        )

Training:   0%|          | 0/52 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

Training:   0%|          | 0/52 [00:00<?, ?it/s]

Training:   0%|          | 0/52 [00:00<?, ?it/s]

Training:   0%|          | 0/52 [00:00<?, ?it/s]

...

Training:   0%|          | 0/52 [00:00<?, ?it/s]

Training:   0%|          | 0/52 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

Training:   0%|          | 0/52 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/22 [00:00<?, ?it/s]

In [16]:
history_path = RESULTS_DIR / f"{CONFIG['experiment_name']}_history.json"
with open(history_path, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2, ensure_ascii=False)

In [17]:
checkpoint = torch.load(best_ckpt_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

predictions, references = run_inference(
    model=model,
    loader=test_loader,
    device=device,
    idx_to_char=idx_to_char,
    vocab=vocab,
)

char_error_rate = cer(references, predictions)
elapsed_total = (time.time() - start_time) / 60

Test inference:   0%|          | 0/22 [00:00<?, ?it/s]

In [18]:
results = {
    "experiment_name": CONFIG["experiment_name"],
    "model_name": CONFIG["model_name"],
    "lang": CONFIG["lang"],
    "data_size": CONFIG["data_size"],
    "num_train_samples": len(train_dataset),
    "num_epochs": CONFIG["num_epochs"],
    "target_iterations": CONFIG["target_iterations"],
    "actual_iterations": total_iterations,
    "best_dev_loss": best_dev_loss,
    "cer": char_error_rate,
    "training_time_minutes": elapsed_total,
}

save_results_json(
    results,
    RESULTS_DIR / f"{CONFIG['experiment_name']}_{len(TRAIN_LANGUAGES)}_results.json",
)

save_predictions_csv(
    references,
    predictions,
    RESULTS_DIR / f"{CONFIG['experiment_name']}_predictions.csv",
)

print(results)

{'experiment_name': 'mhubert_multilingual_swa_10min', 'model_name': 'utter-project/mHuBERT-147-base-3rd-iter', 'lang': 'swa', 'data_size': '10min', 'num_train_samples': 414, 'num_epochs': 116, 'target_iterations': 1500, 'actual_iterations': 1508, 'best_dev_loss': 1.2376684952865948, 'cer': 0.2688424973047143, 'training_time_minutes': 103.09441100358963}
